## 데이터 수집 자동화

### 동적페이지 스크래핑

- 스크패핑하는 웹서버에 부하를 많이 주는 작업
- 웹사이트 오픈, 자동 로그인, 해당페이지로 이동, 필요 내용 검색, 검색결과 스크래핑, 다음 페이지 이동, 검색결과 스크래핑, 반복...

#### 주요 사이트 스크래핑

- 다나와 : 난이도 중 - (상품 검색, 가격비교, 스펙, ...). JS로 페이지 변환으로 명시적 대기로 로딩시간 필요, User-Agent 미체크시 접근 거부

- 네이버 : 난이도 상 - 포털, CSS명을 시간별로 변경, 연속으로 스크래핑 요청시 캡차 호출

- 구글 : 난이도 상 - 포털, 봇 감지, 동적 페이지구조 변경, 연속 스크래핑에 캡차 호출

- 쿠팡 : 난이도 극상 - 쇼핑몰, Anti-bot 솔루션 운영, 봇 접근시 403 에러 발생, 셀레니움으로 접근 IP차단

#### 동적 스크래핑 실습 사이트

- https://quotes.toscrape.com/
- 유명인사들의 명언 등을 보여주는 사이트
- 스크래핑을 위해 개발된 샘플 사이트

In [1]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)
driver.get('https://quotes.toscrape.com/')

footer = wait.until(
    EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
)
    

##### 자동 로그인 연습

- wait.until() : 다음 페이지 화면이 전화됬을때 모두 표시되기전에 스크래핑 시작하면 오류가 발생. 화면이 무조건 다 로드이후 작업 시작하라는 의미

- time.sleep(10) 보다 효율적

In [4]:
# 로그인 링크 찾아서 클릭
login_link = wait.until(
    EC.element_to_be_clickable((By.LINK_TEXT, 'Login'))
)

login_link.click()

In [6]:
# 아이디 패스워드 입력
username_input = wait.until(
    EC.presence_of_element_located((By.ID, 'username'))
)


password_input = driver.find_element(By.ID , 'password')

username_input.send_keys('admin')
password_input.send_keys('admin')


In [7]:
# 로그인버튼 클릭
login_btn = driver.find_element(By.CSS_SELECTOR, 'input[type="submit"]')

login_btn.click()

In [9]:
# 로그인이 완료된 것 확인
wait.until(
    EC.presence_of_element_located((By.LINK_TEXT, 'Logout'))
)

print('로그인 성공 : 작업시작')


로그인 성공 : 작업시작


##### 인용문 찾기

In [12]:
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

len(quotes)

10

In [42]:
# 첫번재 인용문 
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

first = quotes[0]

text = first.find_element(By.CSS_SELECTOR, '.text').text
author = first.find_element(By.CSS_SELECTOR, '.author').text
link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
tag_text = ', '.join(tag.text for tag in tags)


print(text)
print(author)
print(link)
print(tag_text)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
Albert Einstein
http://goodreads.com/author/show/9810.Albert_Einstein
change, deep-thoughts, thinking, world


##### 한 페이지 수집하기

In [39]:
quotes = driver.find_elements(By.CLASS_NAME, 'quote')

datas = []

for first in quotes:
    text = first.find_element(By.CSS_SELECTOR, '.text').text
    author = first.find_element(By.CSS_SELECTOR, '.author').text
    link = first.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
    tags = first.find_elements(By.CSS_SELECTOR, '.tags > a')
    tag_text = ', '.join(tag.text for tag in tags)

    datas.append({
        'text' : text,
        'author' : author,
        'link' : link,
        'tags' : tag_text
    })

In [40]:
df = pd.DataFrame(datas)

df

,text,author,link,tags
0,“The world as we have created it is a process ...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,http://goodreads.com/author/show/1077326.J_K_R...,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,http://goodreads.com/author/show/1265.Jane_Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,http://goodreads.com/author/show/82952.Marilyn...,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,http://goodreads.com/author/show/7617.Andr_Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,http://goodreads.com/author/show/3091287.Thoma...,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,http://goodreads.com/author/show/44566.Eleanor...,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,http://goodreads.com/author/show/7103.Steve_Ma...,"humor, obvious, simile"


##### 다음 페이지 이동

In [44]:
# 다음 페이지 정보 확인
next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
next_button.click()

##### 여러 페이지 수집하기

In [48]:
rows = []

driver.get('http://quotes.toscrape.com/')

for page in range(1,6):
    wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, 'footer'))
    )

    quotes = driver.find_elements(By.CSS_SELECTOR, '.quote')

    for quote in quotes: #한페이지당 10개씩
        text = quote.find_element(By.CSS_SELECTOR, '.text').text
        author = quote.find_element(By.CSS_SELECTOR, '.author').text
        link = quote.find_elements(By.CSS_SELECTOR, 'span > a')[1].get_attribute('href')
        tags = quote.find_elements(By.CSS_SELECTOR, '.tags > a')
        tag_text = ', '.join(tag.text for tag in tags)

        rows.append({
            'text' : text,
            'author' : author,
            'link' : link,
            'tags' : tag_text
        })
    next_button = driver.find_element(By.CSS_SELECTOR, 'li.next a')
    if not next_button:
        break
    next_button.click()


In [49]:
len(rows)

50

In [50]:
df_quotes = pd.DataFrame(rows)

df_quotes

,text,author,link,tags
0,“The world as we have created it is a process ...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,http://goodreads.com/author/show/1077326.J_K_R...,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,http://goodreads.com/author/show/1265.Jane_Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,http://goodreads.com/author/show/82952.Marilyn...,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,http://goodreads.com/author/show/9810.Albert_E...,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,http://goodreads.com/author/show/7617.Andr_Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,http://goodreads.com/author/show/3091287.Thoma...,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,http://goodreads.com/author/show/44566.Eleanor...,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,http://goodreads.com/author/show/7103.Steve_Ma...,"humor, obvious, simile"


#### 중복치 제거

In [52]:
df_quotes = df_quotes.drop_duplicates(subset=['text','author'])

In [53]:
df_quotes.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    50 non-null     str  
 1   author  50 non-null     str  
 2   link    50 non-null     str  
 3   tags    50 non-null     str  
dtypes: str(4)
memory usage: 1.7 KB


In [54]:
# CSV 저장
df_quotes.to_csv('../data/quotes_to_scrap_50.csv', encoding='utf-8')